# Cat vs dog: a CNN from scratch
Load the cats-vs-dogs photos with `image_dataset_from_directory`, scale the pixels, train a three-block CNN,
watch it overfit, add batch normalisation and dropout, and predict two new photos.
Each model is trained with 3 seeds; the Note quotes the means.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"  # take GPU memory only as needed
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "256"   # the GPU is shared; keep scratch memory small
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # Keras notices about tf.data
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers

tf.get_logger().setLevel("ERROR")
HERE = Path.cwd().parent if Path.cwd().name == ".logs" else Path.cwd()   # the Note folder, also from .logs/
(HERE / "data").mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

# The GPU is shared with other jobs. Computing in 16-bit floats (with 32-bit weights) halves the memory
# the activations need; the sigmoid output stays in 32 bits. Speed and memory only: the model is unchanged.
keras.mixed_precision.set_global_policy("mixed_float16")

## 1. The data

In [ ]:
# The Kaggle / Microsoft cats-vs-dogs photos: 12,500 cats and 12,500 dogs, one folder per class.
DATA = Path.home() / "datasets" / "PetImages"
URL = ("https://download.microsoft.com/download/3/E/1/"
       "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
if not DATA.exists():
    zip_path = keras.utils.get_file("kagglecatsanddogs_5340.zip", URL, cache_dir=Path.home(), cache_subdir="datasets")
    import shutil
    shutil.unpack_archive(zip_path, DATA.parent)
# Some files are damaged or are not JPEGs at all; delete them, as the official Keras example does.
removed = 0
for path in DATA.glob("*/*"):
    with open(path, "rb") as f:
        is_jpeg = b"JFIF" in f.peek(10)
    if not is_jpeg:
        path.unlink()
        removed += 1
counts = {d.name: len(list(d.iterdir())) for d in sorted(DATA.iterdir())}
print("removed now:", removed, "| images per class:", counts, "| total:", sum(counts.values()))

In [ ]:
# 80% for training, 20% for validation; every image is resized to 256 x 256 x 3.
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA, labels="inferred", label_mode="int", batch_size=32, image_size=(256, 256),
    validation_split=0.2, subset="both", seed=1337)
print("classes:", train_ds.class_names, "(label 0, label 1)")

In [ ]:
# The original photos come in many sizes: look at 300 of them.
from PIL import Image
files = sorted(DATA.glob("*/*.jpg"))
rng = np.random.default_rng(0)
sizes = np.array([Image.open(f).size for f in rng.choice(files, 300, replace=False)])
print("width  min / median / max:", sizes[:, 0].min(), np.median(sizes[:, 0]), sizes[:, 0].max())
print("height min / median / max:", sizes[:, 1].min(), np.median(sizes[:, 1]), sizes[:, 1].max())
print("distinct sizes among 300 photos:", len({tuple(s) for s in sizes}))

In [ ]:
# Six small thumbnails for the Note's first figure (the only dataset images kept).
picks = [DATA / "Cat" / "1.jpg", DATA / "Cat" / "21.jpg", DATA / "Cat" / "40.jpg",
         DATA / "Dog" / "3.jpg", DATA / "Dog" / "20.jpg", DATA / "Dog" / "39.jpg"]
thumbs, shapes = [], []
for p in picks:
    img = Image.open(p).convert("RGB")
    shapes.append(f"{img.size[0]} x {img.size[1]}")
    thumbs.append(np.array(img.resize((128, 128))))
np.savez_compressed(HERE / "data" / "samples.npz", images=np.array(thumbs),
                    labels=np.array([p.parent.name.lower() for p in picks]), sizes=np.array(shapes))
print(shapes)

## 2. Scale the pixels to 0..1
`process` divides every pixel by 255, as in the Note. The images are cached in memory as whole numbers (uint8)
the first time they are read, so later epochs do not decode the JPEG files again, and reshuffled every epoch.

In [ ]:
def process(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def to_uint8(image, label):          # 4x less memory for the cache
    return tf.cast(image, tf.uint8), label

AUTOTUNE = tf.data.AUTOTUNE
train = (train_ds.unbatch().map(to_uint8).cache()
         .shuffle(20000, seed=0).batch(32).map(process).prefetch(AUTOTUNE))
val = val_ds.map(to_uint8).cache().map(process).prefetch(AUTOTUNE)
x, y = next(iter(train))
print("one batch:", x.shape, "pixel range", float(tf.reduce_min(x)), "to", float(tf.reduce_max(x)), "| labels", y[:8].numpy())

## 3. The CNN: three convolution and pooling blocks, then three dense layers

In [ ]:
def make_model(bn_dropout=False):
    model = keras.Sequential([keras.Input(shape=(256, 256, 3))])
    for filters in (32, 64, 128):
        model.add(layers.Conv2D(filters, kernel_size=(3, 3), padding="valid", activation="relu"))
        if bn_dropout:
            model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D(pool_size=(2, 2), strides=2, padding="valid"))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation="relu"))
    if bn_dropout:
        model.add(layers.Dropout(0.1))
    model.add(layers.Dense(64, activation="relu"))
    if bn_dropout:
        model.add(layers.Dropout(0.1))
    model.add(layers.Dense(1, activation="sigmoid", dtype="float32"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

make_model().summary()

In [ ]:
# Where the parameters are
m = make_model()
conv = sum(l.count_params() for l in m.layers if isinstance(l, layers.Conv2D))
dense = sum(l.count_params() for l in m.layers if isinstance(l, layers.Dense))
first_dense = m.layers[-3].count_params()
print(f"total {m.count_params():,} | convolution {conv:,} | dense {dense:,} | first dense layer {first_dense:,} "
      f"({first_dense / m.count_params():.1%})")
print("batches per epoch:", len(train_ds))

## 4. Train both models, 3 seeds each, 10 epochs

In [ ]:
import time
rows = []
models = {}
for name, flag in (("plain", False), ("bn_dropout", True)):
    for seed in (0, 1, 2):
        keras.utils.set_random_seed(seed)
        model = make_model(flag)
        t = time.time()
        h = model.fit(train, epochs=10, validation_data=val, verbose=0)
        for e in range(10):
            rows.append(dict(model=name, seed=seed, epoch=e + 1, **{k: h.history[k][e] for k in
                        ("accuracy", "val_accuracy", "loss", "val_loss")}))
        print(f"{name:10s} seed {seed}: train acc {h.history['accuracy'][-1]:.3f}  val acc {h.history['val_accuracy'][-1]:.3f}"
              f"  train loss {h.history['loss'][-1]:.3f}  val loss {h.history['val_loss'][-1]:.3f}  ({time.time() - t:.0f} s)")
        if seed == 0:
            models[name] = model
hist = pd.DataFrame(rows).round(4)
hist.to_csv(HERE / "data" / "history.csv", index=False)

In [ ]:
# Means over the 3 seeds
mean = hist.groupby(["model", "epoch"])[["accuracy", "val_accuracy", "loss", "val_loss"]].mean().round(3)
mean["acc_gap"] = (mean.accuracy - mean.val_accuracy).round(3)
print(mean.to_string())

In [ ]:
# The epoch where the mean validation loss is lowest, and the best mean validation accuracy
for name, g in mean.groupby(level="model"):
    g = g.droplevel("model")
    print(f"{name}: lowest val loss at epoch {g.val_loss.idxmin()} ({g.val_loss.min():.3f});"
          f" best val acc {g.val_accuracy.max():.3f} at epoch {g.val_accuracy.idxmax()};"
          f" epoch 10: train {g.accuracy.iloc[-1]:.3f}, val {g.val_accuracy.iloc[-1]:.3f}, gap {g.acc_gap.iloc[-1]:.3f}")

## 5. Predict two new photos (from Wikimedia Commons, not in the dataset)

In [ ]:
import urllib.request
PHOTOS = Path.home() / "datasets" / "photos"

def fetch(url):
    """Download a photo once (Wikimedia asks for a User-Agent) and return its local path."""
    PHOTOS.mkdir(parents=True, exist_ok=True)
    path = PHOTOS / url.split("/")[-1]
    if not path.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "campusx-notes/1.0 (study notes)"})
        path.write_bytes(urllib.request.urlopen(request).read())
    return path

In [ ]:
new = {"cat": fetch("https://upload.wikimedia.org/wikipedia/commons/b/bb/Kittyply_edit1.jpg"),
       "dog": fetch("https://upload.wikimedia.org/wikipedia/commons/b/bd/Golden_Retriever_Dukedestiny01_drvd.jpg")}
for truth, path in new.items():
    img = keras.utils.load_img(path)                              # a PIL image, any size
    print(truth, "original size", img.size)
    img = keras.utils.load_img(path, target_size=(256, 256))      # resized like the training images
    x = keras.utils.img_to_array(img) / 255.0                     # (256, 256, 3), scaled like training
    x = np.expand_dims(x, axis=0)                                 # a batch of one: (1, 256, 256, 3)
    for name, model in models.items():
        p = float(model.predict(x, verbose=0)[0, 0])
        print(f"   {name:10s} P(dog) = {p:.3f} -> {'dog' if p > 0.5 else 'cat'}")